# 10.5 · Experiment tracking with MLflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/05-mlflow/lesson.ipynb)

Read the [README](README.md) first. This notebook logs four churn models to MLflow, compares them, registers the winner as the "champion", and reloads it by name. Runs are stored in `mlruns/` at the course root (git ignores it).

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


In [2]:
import os, logging
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow, mlflow.sklearn
logging.getLogger("mlflow").setLevel(logging.ERROR)     # hide MLflow's informational notices

(ROOT / "mlruns").mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlruns' / 'mlflow.db').as_posix()}")
EXPERIMENT = "shopwave-churn"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(ROOT / "mlruns" / "artifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
print("Tracking runs in mlruns/mlflow.db at the course root")

from mlflow import MlflowClient
client = MlflowClient()
# Start clean, so re-running this notebook does not pile up duplicate runs
for run in mlflow.search_runs(experiment_names=[EXPERIMENT], output_format="list"):
    client.delete_run(run.info.run_id)
if client.search_registered_models(filter_string="name='shopwave-churn'"):
    client.delete_registered_model("shopwave-churn")

Tracking runs in mlruns/mlflow.db at the course root


In [3]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
X_train, X_test, y_train, y_test = train_test_split(
    cust[cm.FEATURES], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, train 2,487 / test 830


## 1. Four candidates, one function

Each run records **what was tried** (parameters), **how well it did** (metrics) and **the model itself** (an artifact). The function below is all the tracking code there is.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

candidates = {
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "tree_depth_3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=42, n_jobs=-1),
    "tree_depth_8": DecisionTreeClassifier(max_depth=8, random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

TRUSTED = ["sklearn.tree._tree.Tree"]        # tree models: we trained them, we trust them (see below)

def train_and_log(name, model):
    with mlflow.start_run(run_name=name) as run:
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
        model.fit(X_train, y_train)
        p = model.predict_proba(X_test)[:, 1]
        params = {k: v for k, v in model.get_params().items()
                  if isinstance(v, (int, float, str, bool)) and "__" not in k or k.endswith("__max_iter")}
        mlflow.log_params({"model": name, "split_seed": 42, **params})
        mlflow.log_metrics({"cv_auc": scores.mean(), "cv_auc_std": scores.std(),
                            "test_auc": roc_auc_score(y_test, p),
                            "test_cost_gbp": cm.expected_cost(y_test, p, bundle["threshold"])})
        mlflow.set_tag("purpose", "lesson 10.5 comparison")
        info = mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3).astype(float),
                                        skops_trusted_types=TRUSTED if "tree" in name or "forest" in name else None)
    return run.info.run_id, info.model_uri

In [5]:
import time
start = time.time()
logged = {name: train_and_log(name, model) for name, model in candidates.items()}
print(f"4 runs logged in {time.time() - start:.0f} s")

4 runs logged in 12 s


## 2. Compare runs

`mlflow.search_runs` returns every run as a DataFrame: the same table the MLflow UI shows. Sort by cross-validated AUC, never by the test AUC (that would be tuning on the test set, 04.1).

In [6]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.cv_auc DESC"])
cols = {"tags.mlflow.runName": "run", "metrics.cv_auc": "cv_auc", "metrics.cv_auc_std": "cv_std",
        "metrics.test_auc": "test_auc", "metrics.test_cost_gbp": "test_cost_£"}
board = runs[list(cols)].rename(columns=cols).round(3)
board

,run,cv_auc,cv_std,test_auc,test_cost_£
0,logistic_regression,0.740,0.029,0.741,3945.0
1,random_forest,0.736,0.030,0.731,3780.0
2,tree_depth_3,0.731,0.024,0.724,4150.0
3,tree_depth_8,0.683,0.020,0.649,5600.0


In [7]:
best = board.iloc[0]
bar = best["cv_auc"] - best["cv_std"]
print(f"one-standard-error bar: {best['cv_auc']:.3f} - {best['cv_std']:.3f} = {bar:.3f}")
print("tied with the best:", list(board.loc[board["cv_auc"] >= bar, "run"]))

one-standard-error bar: 0.740 - 0.029 = 0.711
tied with the best: ['logistic_regression', 'random_forest', 'tree_depth_3']


The same conclusion as 05.10: the reasonable models are tied, and logistic regression is the simplest of them. The deep tree (depth 8) is the one model clearly worse: it memorises.

## 3. Register the champion

The **model registry** gives a run's model a stable name and version numbers. An **alias** such as `champion` points at the version in use. The API would load `models:/shopwave-churn@champion` and never needs to know a run ID.

In [8]:
run_id, model_uri = logged["logistic_regression"]
import contextlib, io
with contextlib.redirect_stderr(io.StringIO()):          # hide MLflow's "Created version" notice
    version = mlflow.register_model(model_uri, "shopwave-churn")
client.set_registered_model_alias("shopwave-churn", "champion", version.version)
print(f"shopwave-churn version {version.version} is now @champion (run {run_id[:8]})")

shopwave-churn version 1 is now @champion (run 5a96af65)


In [9]:
champion = mlflow.sklearn.load_model("models:/shopwave-churn@champion")
same = np.allclose(champion.predict_proba(X_test)[:, 1], bundle["pipeline"].predict_proba(X_test)[:, 1])
print("champion gives the same predictions as the saved bundle:", same)

champion gives the same predictions as the saved bundle: True


## 4. Reproduce a run from its record

Six months later someone asks how the random forest was built. Everything needed is in the run: rebuild it from the logged parameters and check the metric.

In [10]:
rf_run = mlflow.get_run(logged["random_forest"][0])
p = rf_run.data.params
rebuilt = RandomForestClassifier(n_estimators=int(p["n_estimators"]), min_samples_leaf=int(p["min_samples_leaf"]),
                                 random_state=int(p["random_state"]), n_jobs=-1).fit(X_train, y_train)
print(f"logged test AUC {rf_run.data.metrics['test_auc']:.4f}, "
      f"rebuilt {roc_auc_score(y_test, rebuilt.predict_proba(X_test)[:, 1]):.4f}")

logged test AUC 0.7307, rebuilt 0.7307


Identical, because the run recorded the parameters **and** the seeds, and the data split is fixed. Remove `random_state` from the record and this check would fail.

## 5. Safe model files

MLflow 3 saves scikit-learn models in the **skops** format, which, unlike a plain pickle (10.1), checks every type inside the file before loading it. Tree models contain a low-level type it will not load unless you say you trust it, which is why `train_and_log` passes `skops_trusted_types` for the tree models. You trained them, so you can trust them. Never add a type to that list for a file someone sent you.

## 6. The UI

In a terminal at the course root (same on Windows, macOS and Linux):

```bash
python -m mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db
```

Open http://127.0.0.1:5000, choose the `shopwave-churn` experiment, tick runs and press **Compare**. Ctrl+C stops it.

In [11]:
print("runs stored:", len(runs), "| database:", (ROOT / "mlruns" / "mlflow.db").stat().st_size // 1024, "KB")

runs stored: 4 | database: 1048 KB


## Quiz answers

1. Parameters (what you chose), metrics (how it did), artifacts (files: the model, plots), tags (labels such as purpose or author).
2. Tuning or choosing on the test set makes its score optimistic; the test set is for the final check only.
3. A stable name for "the model in production" that can be moved to a new version without changing any code that loads it.
4. Its code version (git commit), the data version, library versions, and every random seed.
5. Unpickling can run arbitrary code; skops checks the types first and refuses unknown ones.

**Next:** [10.6 Monitoring, data drift, concept drift, retraining](../06-monitoring-drift/)